# Universal Context Engine (Converged Edition)

**Copyright 2026, Denis Rothman**

**Goal:** In this notebook we will integrate **Oracle 23ai (Converged Enterprise DB):** agents in a universal content engine to prove that the exact same reasoning architecture (engine.py) can seamlessly switch between:

* **Oracle 23ai (Vector Store):** For **Cyber-Security & Digital Forensics** (Unstructured data from Chapter 2).
* **Oracle 23ai (Relational Table):** For **HR & Recruitment** (Hybrid data from Chapter 3).


# 1.Installation & Setup

In [ ]:
# Install Oracle 23ai and Frozen LLM dependencies
# We lock versions to prevent internal Pydantic/OpenAI parsing conflicts
!pip install --upgrade --force-reinstall \
    openai==2.26.0 \
    pydantic==2.11.9 \
    tiktoken==0.12.0 \
    tqdm==4.67.1 \
    tenacity==9.1.2 \
    oracledb==3.4.1 \
    requests==2.32.4 \
    cryptography==43.0.3 \
    pyOpenSSL==24.2.1 \
    --quiet

print("✅ Oracle 23ai and Frozen LLM dependencies installed.")

✅ Oracle 23ai and Frozen LLM dependencies installed.


In [ ]:
import os
import sys
from google.colab import drive, userdata
from openai import OpenAI

# Mount Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Initialize Oracle Client

try:
    # Explicitly check for the mandatory OpenAI key
    openai_key = userdata.get("API_KEY")
    if not openai_key:
        raise ValueError("OpenAI API_KEY not found in Secrets.")

    os.environ["OPENAI_API_KEY"] = openai_key
    client = OpenAI()
    print("✅ OpenAI initialized (Mandatory).")

except Exception as e:
    print(f"❌ Critical Error: OpenAI initialization failed. {e}")
    # We raise the error here because OpenAI is required for the Planner to work at all
    raise e

✅ OpenAI initialized (Mandatory).


In [ ]:
#@title Retrieving engine library
import time

!curl -L -H "Accept: application/vnd.github.v3.raw" "https://api.github.com/repos/Denis2054/RAG-Driven-Generative-AI-2nd-Edition/contents/commons/engine/engine.zip" -o engine.zip

time.sleep(10)
!unzip -o engine.zip -d /content

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 13119  100 13119    0     0  39943      0 --:--:-- --:--:-- --:--:-- 39875
Archive:  engine.zip
  inflating: /content/agent_archivist.py  
  inflating: /content/agent_recruiter.py  
  inflating: /content/agents.py      
  inflating: /content/engine.py      
  inflating: /content/helpers.py     
  inflating: /content/oracle_lib.py  
  inflating: /content/registry.py    


In [ ]:
# 3. Import Local Modules (Uploaded Files)
try:
    from engine import context_engine
    from registry import AGENT_TOOLKIT
    from oracle_lib import OracleManager
    print("✅ Universal Engine Modules imported.")
except ImportError as e:
    print(f"❌ Module Import Failed: {e}\nUpload engine.py, registry.py, agents.py, helpers.py, oracle_lib.py, agent_archivist.py, agent_recruiter.py")

✅ Universal Engine Modules imported.


In [ ]:
# Initialize Oracle Connection (Enterprise Layer)
try:
    OracleManager.initialize()
    print("✅ Oracle 23ai Connection established.")
except Exception as e:
    print(f"⚠️ Oracle Connection Failed: {e}\n(Ignore if you only want to run other scenarios)")

✅ Oracle 23ai Connection established.


# 2.Initialize the Universal Engine

In [ ]:
# 2. Initialize the Universal Engine Wrapper
from engine import context_engine

# We define a wrapper function that pre-configures the engine with our clients and models.
# This allows us to just pass the 'goal' in the scenario steps below.
def run_universal_engine(goal):
    return context_engine(
        goal=goal,
        client=client,
        generation_model="gpt-5.1",
        embedding_model="text-embedding-3-small"
    )

print("🚀 Universal Context Engine Wrapper is READY.")
# Access the registry dictionary keys directly to see the list of agents
print(list(AGENT_TOOLKIT.registry.keys()))

🚀 Universal Context Engine Wrapper is READY.
['Writer', 'Summarizer', 'OracleArchivist', 'OracleRecruiter']


# 3.Recruitment Scenarios
This execution plan switches to the `OracleRecruiter` agent. The Engine doesn't change code, only the configuration.

In [ ]:
#@title Oracle data verification
from oracle_lib import OracleManager

print("\n=== Oracle Hybrid Table Summary ===\n")

# Use the Manager's context manager instead of a raw 'connection' variable
with OracleManager.get_cursor() as cursor:
    # --- 1. Check HR Tables ---
    tables = ['CANDIDATE_POOL', 'RECRUITMENT_RULES']

    for t in tables:
        try:
            cursor.execute(f"SELECT COUNT(*) FROM {t}")
            result = cursor.fetchone()
            if result:
                print(f"Table {t}: {result[0]} rows")
        except Exception as e:
            print(f"Table {t}: Not Found or Error ({e})")

    print("\n--- Sample Candidate (Structured + Vector) ---")
    try:
        cursor.execute("""
            SELECT candidate_id, full_name, salary_expectation,
                   CASE WHEN resume_vector IS NOT NULL THEN '✅ Vector Present' ELSE '❌ No Vector' END
            FROM candidate_pool
            FETCH FIRST 2 ROWS ONLY
        """)
        for row in cursor.fetchall():
            print(row)
    except Exception as e:
        print(f"Error fetching candidates: {e}")

print("\n=== Verification Complete ===")


=== Oracle Hybrid Table Summary ===

Table CANDIDATE_POOL: 5 rows
Table RECRUITMENT_RULES: 3 rows

--- Sample Candidate (Structured + Vector) ---
('CAND_005', 'Quinn R.', 155000, '✅ Vector Present')
('CAND_003', 'Casey M.', 210000, '✅ Vector Present')

=== Verification Complete ===


In [ ]:
import textwrap

def run_oracle_recruitment_scenario(role: str, width: int = 80):
    """
    Encapsulates Scenario A: Recruitment using the OracleRecruiter agent.

    Args:
        role (str): The job title or role description to search for.
        width (int): The max line length for the output display (default 80).

    Returns:
        tuple: (result, trace)
    """
    # 1. Define the simplified goal based only on the role
    recruitment_goal = role

    # 2. Visual separation for the start
    print("-" * width)
    print(f"Goal: {recruitment_goal}")
    print("-" * width)

    try:
        # 3. Execute the engine
        # Assuming run_universal_engine is defined in global scope
        result, trace = run_universal_engine(recruitment_goal)

        # 4. Present the text results nicely
        print("\n--- FINAL RECRUITMENT EMAILS ---\n")

        # Split by lines to preserve paragraph structure, then wrap lines individually
        if result:
            for line in str(result).splitlines():
                if line.strip(): # If line is not empty
                    print(textwrap.fill(line, width=width))
                else:
                    print() # Preserve empty lines
        else:
            print("No result returned.")

        print("-" * width)
        return result, trace

    except NameError:
        print("Error: `run_universal_engine` is not defined.")
        return None, None

In [ ]:
#@title Scenario A : Searching for experienced Python developers
# The text will now automatically wrap at 80 characters for better readability
res, log = run_oracle_recruitment_scenario("Find Experienced Python Developers with experience")

--------------------------------------------------------------------------------
Goal: Find Experienced Python Developers with experience
--------------------------------------------------------------------------------

--- FINAL RECRUITMENT EMAILS ---

**Overview of Talent Pool**

The current set of candidates is heavily weighted toward leadership, project
management, and non-Python back-end development. Only one candidate (CAND_005)
appears to have a strong Python background appropriate for an “Experienced
Python Developer” role; the others may be more suitable for adjacent roles
(project management, Java/PL-SQL backend, or leadership/EM roles) rather than
hands-on Python development.

---

- **Candidate: CAND_003 – Casey M.**
  - **Years of Python Experience:** 0 (no coding / Python experience)
  - **Frameworks/Libraries:** Not applicable
  - **Domain Experience:**
    - High‑stakes government contracts
    - Project management across SDLC, risk mitigation, stakeholder communication

In [ ]:
#@ Scenario B Searching for experienced Python developers and writing an email
# The text will now automatically wrap at 80 characters for better readability
res, log = run_oracle_recruitment_scenario("Find Experienced Python Developers with experience and 250000 salary max. Then, write a job interview email to the top candidate, explicitly using their name.")

--------------------------------------------------------------------------------
Goal: Find Experienced Python Developers with experience and 250000 salary max. Then, write a job interview email to the top candidate, explicitly using their name.
--------------------------------------------------------------------------------

--- FINAL RECRUITMENT EMAILS ---

Subject: Interview Invitation – Python Developer Role at [Company Name]

Dear Quinn R.,

Based on your strong Python background and eight years of engineering leadership
experience, we were very impressed with your profile. Your mix of hands-on
Python expertise and experience mentoring teams makes you an outstanding fit for
our Python-focused role and culture at [Company Name], Quinn R.

We would like to invite you to a formal interview to discuss the position and
your experience in more detail. Please choose a time that works best for you
from the following options, or use this link to select another slot: [Scheduling
Link Placeh

In [ ]:
close=False
if close==True:
 OracleManager.close()